In [1]:
import pandas as pd

In [3]:
Data = 'Data/ratings.csv'

df = pd.read_csv(Data, sep=',')
df = df.drop(columns=['timestamp'])

In [10]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import GridSearchCV, train_test_split


reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df, reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [5]:
model = SVD(n_factors=20, random_state=42)
model.fit(trainset) 

In [6]:
predictions = model.test(testset)  
rmse = accuracy.rmse(predictions)  

RMSE: 0.8780


In [8]:
# Predict a single (user, movie) pair directly
user_id = 1
movie_id = 2253
pred = model.predict(user_id, movie_id)
print(pred)

user: 1          item: 2253       r_ui = None   est = 3.58   {'was_impossible': False}


In [14]:
param_grid = {
    'n_factors': [20, 50, 100],
    'n_epochs': [20, 30],
    'lr_all': [0.002, 0.005, 0.01],
    'reg_all': [0.02, 0.1, 0.4]
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)

print(f"Best RMSE: {gs.best_score['rmse']:.4f}")
print(f"Best params: {gs.best_params['rmse']}")

Best RMSE: 0.8625
Best params: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


In [ ]:
best_params = gs.best_params['rmse']
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

final_model = SVD(**best_params, random_state=42)
final_model.fit(trainset)

predictions = final_model.test(testset)
final_rmse = accuracy.rmse(predictions)